# Claude 4.6 Opus — Zero-Shot Anomaly Detection (Probability Scoring) — TelecomTS

> Labels were produced via direct numerical inference: the model received only raw KPI time series serialized as text and output a continuous anomaly probability score (0.0–1.0) per sample. No API, no precomputed statistics, no metadata.

## Model

| Model | Family | Input | Output |
|-------|--------|-------|--------|
| **claude-4.6-opus** | Anthropic | Raw KPI time series only | Anomaly probability (0.0–1.0) |

## Experiment Design

- **Input:** Raw KPI time series only (128 timesteps × 16 KPIs per sample).
- **No labels, no precomputed statistics, no descriptions** were provided.
- **Output:** Continuous anomaly probability score (0.0 = normal, 1.0 = anomaly).
- **Binary threshold:** 0.5 for hard label metrics (Precision, Recall, F1).
- **Ranking metrics:** AUROC and AP computed from continuous scores.

## Dataset

| Split | Samples | Anomaly | Normal | Balance |
|-------|---------|---------|--------|---------|
| Test  | 494     | 247     | 247    | 50/50   |

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    matthews_corrcoef, roc_curve, precision_recall_curve,
)

SEED = 42
np.random.seed(SEED)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Imports ready.")

## 2. Load Data

In [ ]:
pred_file = Path("test_without_label/TelecomTS_test_predicted.npz")
gt_file = Path("data/TelecomTS_test.npz")

pred_data = np.load(pred_file, allow_pickle=True)
gt_data = np.load(gt_file, allow_pickle=True)

y_score = pred_data["y_score"].astype(np.float64)
y_pred = pred_data["y_pred"].astype(np.int64)
y_true = gt_data["y"].astype(np.int64)

print(f"Ground truth: {len(y_true)} samples, {y_true.sum()} anomalies, {(y_true == 0).sum()} normal")
print(f"Predictions (threshold=0.5): {y_pred.sum()} anomalies, {(y_pred == 0).sum()} normal")
print(f"Score range: [{y_score.min():.4f}, {y_score.max():.4f}], mean={y_score.mean():.4f}")

assert len(y_true) == len(y_pred) == len(y_score), "Mismatch!"

pred_data.close()
gt_data.close()

## 3. Compute All Metrics

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
accuracy = accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0.0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
balanced_accuracy = (recall + specificity) / 2.0

# Continuous-score metrics
auroc = roc_auc_score(y_true, y_score)
ap = average_precision_score(y_true, y_score)

print("=" * 60)
print("  claude-4.6-opus — Zero-Shot Anomaly Detection (TelecomTS)")
print("=" * 60)
print(f"  Precision          : {precision:.4f}")
print(f"  Recall (TPR)       : {recall:.4f}")
print(f"  F1 Score           : {f1:.4f}")
print(f"  Accuracy           : {accuracy:.4f}")
print(f"  Balanced Accuracy  : {balanced_accuracy:.4f}")
print(f"  Specificity (TNR)  : {specificity:.4f}")
print(f"  NPV                : {npv:.4f}")
print(f"  FPR                : {fpr_val:.4f}")
print(f"  FNR                : {fnr:.4f}")
print(f"  MCC                : {mcc:.4f}")
print(f"  AUROC              : {auroc:.4f}")
print(f"  AP                 : {ap:.4f}")
print(f"{'=' * 60}")
print(f"\n  Confusion Matrix (threshold=0.5):")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print()
print(classification_report(y_true, y_pred, target_names=["Normal", "Anomaly"], digits=4))

## 4. ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, _ = roc_curve(y_true, y_score)
ax = axes[0]
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'Claude 4.6 Opus (AUROC={auroc:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# PR
prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_score)
ax = axes[1]
ax.plot(rec_arr, prec_arr, color='darkorange', lw=2, label=f'Claude 4.6 Opus (AP={ap:.3f})')
ax.axhline(y=y_true.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({y_true.mean():.2f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'claude_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(cm, cmap="Blues", interpolation="nearest")
ax.set_title("Confusion Matrix (counts)", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "Anomaly"]); ax.set_yticklabels(["Normal", "Anomaly"])
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=18, fontweight="bold", color=color)
fig.colorbar(im, ax=ax, shrink=0.8)

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ax = axes[1]
im = ax.imshow(cm_norm, cmap="Blues", interpolation="nearest", vmin=0, vmax=1)
ax.set_title("Confusion Matrix (normalized)", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "Anomaly"]); ax.set_yticklabels(["Normal", "Anomaly"])
for i in range(2):
    for j in range(2):
        color = "white" if cm_norm[i, j] > 0.5 else "black"
        ax.text(j, i, f"{cm_norm[i, j]:.2%}", ha="center", va="center", fontsize=16, fontweight="bold", color=color)
fig.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'claude_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_score[y_true == 0], bins=30, alpha=0.6, label='Normal', color='steelblue', edgecolor='black')
ax.hist(y_score[y_true == 1], bins=30, alpha=0.6, label='Anomaly', color='tomato', edgecolor='black')
ax.axvline(x=0.5, color='black', linestyle='--', lw=2, label='Threshold (0.5)')
ax.set_xlabel('Anomaly Probability Score', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Claude 4.6 Opus — Score Distribution by True Class', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'claude_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Summary CSV

In [ ]:
summary = pd.DataFrame([{
    'Model': 'claude-4.6-opus',
    'Precision': precision, 'Recall': recall, 'F1': f1,
    'Accuracy': accuracy, 'Bal. Acc.': balanced_accuracy,
    'Specificity': specificity, 'AUROC': auroc, 'AP': ap, 'MCC': mcc,
    'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
}])
summary.to_csv(RESULTS_DIR / 'claude_metrics_summary.csv', index=False)
print(summary.to_string(index=False))
print(f"\nSaved to {RESULTS_DIR / 'claude_metrics_summary.csv'}")